In [3]:
from dotenv import load_dotenv, find_dotenv
import pandas as pd
import os
import seaborn as sns
import pandas as pd

load_dotenv(find_dotenv())

path = os.getenv('DATA_SOURCE')

In [8]:
# Read data from csv file and create DataFrame:
csv_data = pd.read_csv(path,sep=';')
df = pd.DataFrame(csv_data)

ValueError: Invalid file path or buffer object type: <class 'NoneType'>

In [6]:
# Dataset size:
shape = df.shape
print(f"Row count: {shape[0]}")
print(f"Column count: {shape[1]}")

NameError: name 'df' is not defined

In [ ]:
# Data type of each column:
data_type = df.info()
print(data_type)

## Data Issues:

- Column **MonthlyCharges & TotalCharges**: Đang là `string` → cần kiểm tra dữ liệu và convert sang numeric  
- Column **avg_monthly_gb**: Kiểu `object` → cần inspect và chuẩn hóa (có thể chứa dữ liệu không nhất quán)

In [ ]:
# Quick view first and last 5 data rows:
head = df.head()
tail = df.tail()
print(f"First 5 rows: \n", head)
print(f"Last 5 rows: \n", tail)

In [ ]:
# Data distribution:

# For numerical columns:
desc_num = df.describe()
print("Describing numerical columns:")
print(desc_num)

In [ ]:
# For categorical columns:
desc_cate = df.select_dtypes(include="object")
print("Describing categorical columns:")

for col_name, col_data in desc_cate.items():
    print(col_name)
    print(f"Value distribution: {col_data.value_counts()}") # Count records of each type
    print(f"Count of unique value: {col_data.nunique()}") # Count unique values
    print(f"Most frequent data: {col_data.mode()[0]}") # Define most frequent data
    print("\n\n")

In [ ]:
# Detect rows that are fully duplicated:
duplicate = df.duplicated().sum()
print(duplicate)

In [1]:
##II. Process missing value data
#II.1. Kiểm tra các cột có giá trị Na -- Detect column with Na value
na_columns = df.columns[df.isna().any()]
na_info = pd.DataFrame({
    'No of Na': df[na_columns].isna().sum(),
    'Type of data': df[na_columns].dtypes})
print("The columns with Na values are:")
print(na_info)

# II.2. Xử lý cột có Na cần chuẩn hóa cột avg_monthly_gb (thay thế các case dữ liệu sai ký tự ngăn cách từ , thành . và biến thành float)
# Standardize the column avg_monthly_gb to replace Na value
df['avg_monthly_gb'] = df['avg_monthly_gb'].astype(str).str.replace(r'\.(?=\d{3})', ',', regex=True)
df['avg_monthly_gb'] = pd.to_numeric(df['avg_monthly_gb'], errors='coerce')
print(df['avg_monthly_gb'])

# II.3. Nhận diện các cột phân bố giá trị không đồng đều để lựa chọn sử dụng thay giá trị Na của cột thành mean hay median
# Visualize to identify the distribution of data column & replace Na with mean/ median
cols_to_plot = ['annual_income', 'customer_satisfaction', 'num_complaints', 'credit_score', 'avg_monthly_gb']
plt.figure(figsize=(16, 12))
sns.set_style("whitegrid")
for i, col in enumerate(cols_to_plot, 1):
    plt.subplot(3, 2, i)
    sns.histplot(df[col].dropna(), kde=True, color='skyblue', bins=30)
    plt.title(f'Value distribution of {col}')
    plt.xlabel('Value')
    plt.ylabel('Frequency')
plt.tight_layout()
plt.show()

## Các cột sử dụng mean: 'customer_satisfaction'
## Các cột sử dụng median: 'annual_income', 'credit_score', 'avg_monthly_gb',  'num_complaints'

median_val = df['customer_satisfaction'].median()
df['customer_satisfaction'] = df['customer_satisfaction'].fillna(median_val)

cols_median = ['annual_income', 'credit_score', 'avg_monthly_gb', 'num_complaints']
for col in cols_median:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

# II.4. Kiểm tra lại sau khi xử lý còn cột nào chứa giá trị Na không -- Recheck whether any Na value in the data
print(df.isna().sum())

NameError: name 'df' is not defined